# Withdrawal risk: the same model in a notebook

Talk track: *Some of your team will live in VS Code, some in notebooks. This is the same model in a Snowflake notebook running on the compute pool, reading the same Feature Store dataset, logging to the same experiment.* Features are synthetic demo data.

## 1. The training data, straight from the Feature Store

Talk track: *This is the versioned dataset the Feature Store produced: the label plus the WITHDRAWAL_SIGNALS feature view. Every model trained on it shows up in lineage.*

In [ ]:
%%sql -r training_data
SELECT * FROM UOFR_XML_DEMO.ML_FEATURES.WITHDRAWAL_SIGNALS$V1 LIMIT 10

## 2. Load the dataset and train

Talk track: *Same libraries, same parameters as train.py. XGBoost is already in the runtime image, nothing to install.*

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.dataset import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

session = get_active_session()
FEATURES = ["GPA", "COURSE_LOAD", "MIDTERM_GRADE_POINTS", "LMS_LOGINS_LAST_14D",
            "FINANCIAL_HOLD", "FIRST_GENERATION", "ADVISING_VISITS"]
PARAMS = {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.08, "subsample": 0.9}

ds = load_dataset(session, "UOFR_XML_DEMO.ML_FEATURES.WITHDRAWAL_TRAINING", "V1")
df = ds.read.to_pandas()            # 15k rows: small enough to train in memory
X, y = df[FEATURES].astype(float), df["WITHDREW"].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = XGBClassifier(**PARAMS, eval_metric="auc", random_state=42).fit(X_tr, y_tr)
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"AUC on held-out enrollments: {auc:.3f}")

## 3. Log this run to the experiment

Talk track: *Notebook runs land in the same experiment as the VS Code and compute-pool runs, so you compare them side by side in AI & ML, Experiments.*

In [ ]:
from datetime import datetime
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session, database_name="UOFR_XML_DEMO", schema_name="ML")
exp.set_experiment("UR_WITHDRAWAL_RISK")
with exp.start_run(f"NOTEBOOK_{datetime.now():%Y%m%d_%H%M%S}"):
    exp.log_params({**PARAMS, "compute": "notebook", "dataset": "WITHDRAWAL_TRAINING V1"})
    exp.log_metrics({"auc": float(auc)})
print("Logged to experiment UR_WITHDRAWAL_RISK")

## 4. What drives the prediction

Talk track: *Feature importance from the model: midterm grade and financial hold dominate, which is how the synthetic data was built. On your data this is where the advising conversation starts.*

In [ ]:
import matplotlib.pyplot as plt
imp = sorted(zip(FEATURES, model.feature_importances_), key=lambda t: t[1])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh([f for f, _ in imp], [v for _, v in imp], color="#29B5E8")
ax.set_title("Feature importance (XGBoost gain)")
plt.tight_layout()
plt.show()

## 5. Score with the registered model, in SQL

Talk track: *And the production model, the default version in the registry, scored in SQL right here. FERPA masking still applies to whoever runs this.*

In [ ]:
%%sql -r top_risk
WITH mv AS MODEL UOFR_XML_DEMO.ML.UR_WITHDRAWAL_RISK
SELECT f.student_id, f.course_code, f.gpa, f.midterm_grade_points, f.financial_hold,
       ROUND(mv!PREDICT_PROBA(f.gpa, f.course_load, f.midterm_grade_points, f.lms_logins_last_14d,
             IFF(f.financial_hold, 1, 0), IFF(f.first_generation, 1, 0), f.advising_visits):output_feature_1::FLOAT, 3) AS withdrawal_risk
FROM UOFR_XML_DEMO.ML.WITHDRAWAL_FEATURES f
ORDER BY withdrawal_risk DESC
LIMIT 10